# Notebook to resample the raw data of the FBG

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

## Processing

### Parameters

In [ ]:
ncfiles = glob.glob('/mnt/ICELAND_2024/GUNNUVHER/FBG/LowFreqSampling/aligned_nc_converted/*.nc')
channel = 1 ## Be aware that the channel number is 0-indexed, so channel 1 is actually the second channel in the data.
dt = '10s'

### Importing the data

In [ ]:
#ncfiles =glob.glob('/mnt/ICELAND_2024/GUNNUVHER/FBG/LowFreqSampling/aligned_nc_converted/FBG_2025.01*nc')+ glob.glob('/mnt/ICELAND_2024/GUNNUVHER/FBG/LowFreqSampling/aligned_nc_converted/FBG_2025.02*nc') 
ncfiles.sort()
for i, file in enumerate(tqdm(ncfiles)) :
    ds = xr.open_dataset(file)
    ds = ds.drop_vars(['relative_Time', 'Temperature_calibration','Strain_calibration'])
    #ds = preprocess(ds)
    ds = ds.sel(Channel=channel)
    if i == 0 :
        dsch_hstrain = ds.copy()
        continue
    dsch_hstrain = xr.concat([dsch_hstrain, ds], dim='Time')

100%|██████████| 347/347 [03:09<00:00,  1.83it/s]


### Cleaning the data

In [ ]:
dsch_hstrain = dsch_hstrain.sortby('Time')
dsch_hstrain = dsch_hstrain.resample(Time=dt).median()

### Exporting the data

In [ ]:
dsch_hstrain.to_netcdf('fbg_data_r/gunnvher_channel_%i_fbg_cleaned_%s.nc'%(channel, dt))